# 02. Grad-CAM: ResNet50 và ViT-B/16

So sánh Grad-CAM theo **logit lớp ground-truth**, kể cả dự đoán sai.
ResNet dùng `layer4[-1]`, ViT dùng `encoder.layers[11].ln_1` trước self-attention cuối.
Giữ weights ResNet50 IMAGENET1K_V2 và ViT-B/16 IMAGENET1K_V1 cùng preprocessing chính thức.
Hai crop khác nhau được căn chỉnh trên vùng giao trong tọa độ ảnh gốc.
Lưới chính 7×7, kiểm tra độ nhạy 14×14; metrics: Spearman, Top-20% IoU và entropy chuẩn hóa.

Notebook chạy độc lập: setup → chọn ảnh → chạy/resume → phân tích → export.
`experiments/gradcam/`: `methods.py` (heatmap), `experiment.py` (cohort/checkpoint),
`analysis.py` (metrics/bảng/hình/export). Tái sử dụng `src` và giữ nguyên các module chung.

## 1. Environment and synchronization

Colab: open this notebook from the personal public repository and select a GPU runtime.
Run the setup cell to clone/pull `feat/gradcam`, mount Drive, prepare data and install dependencies.
No GitHub token is needed. On Windows, run `.\sync.ps1` after editing, then rerun setup here.

Input: `MyDrive/cv_2026/data_10k.zip` containing 14 Parquet shards under `data_10k/`.
Setup extracts the shards to `/content` once per runtime and verifies the manifest and IDs/labels.
Code always comes from Git; set `DATA_DIR` only to override the extracted dataset.
Results/checkpoints stay under `MyDrive/cv_2026/metrics/` and `MyDrive/cv_2026/figures/`.
Weights are cached under `MyDrive/cv_2026/torch-cache/`.

Stop an active run before pulling. When available, autoreload refreshes Python modules before
subsequent cells; rerun configuration and experiment cells to recreate objects. If the runtime
cannot load autoreload, setup continues; restart the Python session after code changes. Restart the runtime session when
changing model classes or dependencies, or if an old object still behaves unexpectedly.
Git pull updates Python files, not the notebook cells already open in the Colab editor.
Reopen this notebook from GitHub when its cells change.

Local: use the local repository and dataset; no Drive mount or automatic dependency installation.
See [workflow guide](../docs/personal_remote_colab.md).


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import runpy

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab defaults are ready to use. Local execution keeps the local repository.
PROJECT_DIR = None
DATA_DIR = None
RESULTS_DIR = None
PROJECT_ZIP = "cv_2026/data_10k.zip"  # Input ZIP relative to MyDrive.
MOUNT_DRIVE = True
INSTALL_DEPENDENCIES = IN_COLAB
REPO_URL = "https://github.com/thuhang012/cnn-vit-interpretability-personal.git"
BRANCH = "feat/gradcam"

if IN_COLAB:
    checkout = Path(PROJECT_DIR or "/content/cnn-vit-interpretability").resolve()

    def git(*args):
        return subprocess.check_output(["git", *args], text=True).strip()

    if (checkout / ".git").is_dir():
        if git("-C", str(checkout), "remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("Checkout points to a different repository.")
        if git("-C", str(checkout), "branch", "--show-current") != BRANCH:
            raise RuntimeError("Checkout is on a different branch.")
        if git("-C", str(checkout), "status", "--porcelain"):
            raise RuntimeError("Save Colab checkout edits separately before pulling.")
        git("-C", str(checkout), "pull", "--ff-only", "origin", BRANCH)
    else:
        git("clone", "--branch", BRANCH, REPO_URL, str(checkout))
    print("Code commit:", git("-C", str(checkout), "rev-parse", "--short", "HEAD"))
    configure = runpy.run_path(str(checkout / "scripts/colab_setup.py"))["setup"]
    ROOT, DATA_DIR, RESULTS_DIR = configure(
        checkout, data_dir=DATA_DIR, results_dir=RESULTS_DIR,
        project_zip=PROJECT_ZIP, mount_drive=MOUNT_DRIVE,
        install_dependencies=INSTALL_DEPENDENCIES,
    )
else:
    default_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    ROOT = Path(PROJECT_DIR).expanduser().resolve() if PROJECT_DIR else default_root.resolve()

    def project_path(value, default):
        path = Path(value).expanduser() if value is not None else ROOT / default
        return (path if path.is_absolute() else ROOT / path).resolve()

    DATA_DIR = project_path(DATA_DIR, "data/imagenet/data")
    RESULTS_DIR = project_path(RESULTS_DIR, "results")
    if INSTALL_DEPENDENCIES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                        str(ROOT / "requirements.txt")], check=True)

if not (ROOT / "configs/analysis_set.json").is_file():
    raise FileNotFoundError("PROJECT_DIR must point to the repository root.")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dataset folder is missing: {DATA_DIR}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Reload edited Python modules on subsequent cells after pulling new code.
try:
    shell = get_ipython()
except NameError:
    shell = None
AUTORELOAD_AVAILABLE = False
if shell is not None:
    try:
        shell.run_line_magic("load_ext", "autoreload")
        shell.run_line_magic("autoreload", "2")
        AUTORELOAD_AVAILABLE = True
    except ImportError as error:
        print(f"Optional autoreload is unavailable ({error}). Setup can continue.")
        print("After pulling Python code changes, restart the Python session and rerun cells to load the new code.")
print(f"Environment: {'Colab' if IN_COLAB else 'local'} | Project: {ROOT}")
print(f"Data: {DATA_DIR}\nResults/checkpoints: {RESULTS_DIR}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from experiments.gradcam.experiment import prepare_cohorts, experiment_stores
from experiments.gradcam.analysis import analyze, show_results, plot_summary, plot_examples, export_final

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

In [ ]:
RUN_EXPERIMENT = True      # False: only read existing checkpoints.
SAVE_FINAL = False         # Preview until results have been reviewed.
CHECK_PINNED_DATA = False  # Optional full source/hash verification.
ANALYSIS_SIZE = 10000      # 5000: class-balanced trial; 10000: shared cohort.
SEED = 42
BATCH_SIZE = 1             # Explicit batch; no pilot or automatic batch selection.
ALLOW_DOWNLOAD = IN_COLAB    # Opt in if pretrained weights are missing.
RUN_TAG = f"gradcam_v1_n{ANALYSIS_SIZE}"
CONDITION = "E1"

METRICS_ROOT = RESULTS_DIR / "metrics/experiments/gradcam" / RUN_TAG
OUT = METRICS_ROOT / "intermediate"
FINAL = METRICS_ROOT / "final"
FIG = RESULTS_DIR / "figures/experiments/gradcam" / RUN_TAG / "final"
stores = result = None
figures = {}

## 2. Manifest và cohort

Giữ nguyên manifest 10.000 ảnh (10 ảnh/lớp), seed và cách ghép dự đoán baseline theo ID.
Nhóm đúng/sai vẫn dựa trên baseline của cả ResNet và ViT; không cần chạy lại baseline.
Grad-CAM dùng 10.000 ảnh mặc định; tùy chọn 5.000 ảnh cân bằng lớp chỉ là lượt chạy thử.

In [ ]:
if CHECK_PINNED_DATA:
    # The existing checker uses the dataset path in configs/analysis_set.json.
    if DATA_DIR != (ROOT / "data/imagenet/data").resolve():
        raise ValueError("CHECK_PINNED_DATA uses the configured dataset; run it with the default DATA_DIR.")
    subprocess.run([sys.executable, str(ROOT / "src/create_analysis_manifest.py"), "--check"],
                   cwd=ROOT, check=True)
selection = prepare_cohorts(ROOT, ANALYSIS_SIZE, SEED)
display(selection.groupby("group").size().rename("n_images").to_frame())
print(f"{CONDITION}: {len(selection)} images | seed={SEED}")

## 3. Chạy hoặc tiếp tục checkpoint

Không yêu cầu pilot. Batch mặc định 1; tăng `BATCH_SIZE` khi phù hợp bộ nhớ.
Các kiểm tra logits, heatmap và hook vẫn được giữ trong lúc trích xuất.
ID đã commit không chạy lại; checkpoint khác cấu hình bị từ chối.
OOM dừng lượt chạy, không tự giảm cohort. Chọn run ID mới khi thay code/cấu hình/batch.
Để xem lại, đặt `RUN_EXPERIMENT=False` và `SAVE_FINAL=False`, giữ cùng run ID.

Completed paired checkpoints are reused automatically after checking code, cohort, manifest
and baseline compatibility. Notebook-only commits do not force inference. Original provenance
is preserved. Incomplete runs still use the original strict resume checks; no checkpoint is overwritten.
Final export is disabled by default while reviewing results; Run all includes section 7.

In [ ]:
if not isinstance(BATCH_SIZE, int) or isinstance(BATCH_SIZE, bool) or BATCH_SIZE < 1:
    raise ValueError("BATCH_SIZE must be a positive integer")

# Read completed checkpoints before considering inference. A notebook-only commit
# changes git_revision but does not change the CAMs already computed.
from experiments.gradcam.experiment import RunStore, checksum
import hashlib

cached_stores = []
expected_ids = set(selection.image_id)
expected_records = hashlib.sha256(selection.sort_values("image_id").to_csv(index=False).encode()).hexdigest()
expected_config = {
    "experiment": CONDITION, "seed": SEED, "records": expected_records,
    "target": "ground_truth_logit", "preprocessing": "official",
    "manifest": checksum(ROOT / "data/manifests/analysis_10000_seed42.csv"),
    "dataset_config": checksum(ROOT / "configs/analysis_set.json"),
    "code": {name: checksum(ROOT / "experiments/gradcam" / name)
             for name in ("methods.py", "experiment.py", "analysis.py")},
    "shared_code": {name: checksum(ROOT / "src" / name)
                    for name in ("models.py", "data.py", "evaluation.py", "evaluate_baseline.py")},
    "baseline": {name: checksum(ROOT / f"results/metrics/baseline_notebook/full/{name}_predictions.csv")
                 for name in ("resnet50", "vit_b_16")},
}
for model_name in ("resnet50", "vit_b_16"):
    checkpoint_dir = OUT / CONDITION / model_name
    cached = RunStore(checkpoint_dir) if (checkpoint_dir / "config.json").is_file() else None
    if cached is not None:
        mismatches = [key for key, value in {**expected_config, "model": model_name}.items()
                      if cached.config.get(key) != value]
        if set(cached.config["image_ids"]) != expected_ids:
            mismatches.append("image_ids")
        if mismatches:
            raise ValueError(f"{model_name}: checkpoint differs in {', '.join(mismatches)}. "
                             "Keep the existing run unchanged; choose a new RUN_TAG for a different experiment.")
    cached_stores.append(cached)

if all(store is not None and store.done == expected_ids for store in cached_stores):
    stores = cached_stores
    for store in stores:
        print(f"{CONDITION}/{store.config['model']}: reused {len(store.done)}/{len(expected_ids)} "
              f"saved CAMs; original commit={store.config.get('git_revision', 'unknown')}; no inference or writes")
else:
    batch_sizes = {name: BATCH_SIZE for name in ("resnet50", "vit_b_16")} if RUN_EXPERIMENT else None
    stores = experiment_stores(ROOT, selection, CONDITION, OUT, batch_sizes=batch_sizes,
                               device=DEVICE, seed=SEED, allow_download=ALLOW_DOWNLOAD,
                               data_dir=DATA_DIR)
result = analyze(*stores) if stores is not None else None


## 4. Chỉ số và độ bao phủ

Bảng tổng hợp giữ nguyên nội dung: chỉ số chính, nhóm đúng/sai và coverage.
Heatmap lỗi/suy biến được gắn cờ; metric không xác định giữ NaN, không thay bằng 0.
Không hiển thị bảng per-image hoặc danh sách ID đầy đủ; các bảng này có trong file export.

In [ ]:
if result is not None:
    show_results({CONDITION: result})
else:
    print(f"{CONDITION} chưa hoàn tất; xem tiến độ ở phần trên.")

## 5. Visualization

Giữ nguyên biểu đồ định lượng, cách chọn ảnh theo nhóm baseline và các panel heatmap.
Mỗi input/overlay dùng cùng crop chính thức; tiêu đề ghi rõ model và phương pháp.

In [ ]:
figures = {}
if result is not None:
    figures = {f"{CONDITION}_metrics": plot_summary(result, CONDITION),
               f"{CONDITION}_examples": plot_examples(ROOT, stores, selection, data_dir=DATA_DIR)}
    plt.show()

## 6. Export riêng cho E1

Không cần chạy notebook còn lại. Lưu `metrics.csv`, `per_image.csv`,
CSV dự đoán của hai model và `metadata.json` vào `FINAL`;
hai hình `E1_metrics.png`, `E1_examples.png` vào `FIG`.
Giữ schema số liệu cũ; tên thí nghiệm trong metadata/CSV là `gradcam`.

Accuracy dùng dự đoán trích xuất thành công, kèm số ảnh yêu cầu/lỗi; đây là kiểm tra
tính nhất quán dự đoán, không phải thước đo chất lượng heatmap.
Không ghi đè final hoặc chuyển checkpoint cũ từ notebook chung.
Trên Colab, checkpoint và final đều được lưu trên Drive; khi ngắt phiên có thể mất
phần batch chưa commit, nhưng lượt chạy sau tiếp tục từ các chunk đã lưu.
Notebook được lưu không kèm output nhúng.

In [ ]:
if SAVE_FINAL and result is not None:
    accuracy = export_final(stores, {CONDITION: result}, figures, FINAL, FIG)
    print("Prediction accuracy: percentages on successful extractions; failures listed separately")
    display(accuracy.round(4))
elif not SAVE_FINAL:
    print("Preview only: no final files written.")
else:
    print(f"Final export skipped: {CONDITION} is incomplete.")

## 7. Supplement: connected components of top-20% CAM (preview only)

Read existing E1 checkpoints; no GPU inference, checkpoint updates or result files are written.
Run all reuses completed compatible CAMs and skips final export by default.
To review only this supplement, run the setup cell
and the cells below, skipping the earlier experiment/export cells. Set `COMPONENT_RUN_TAG`
to the completed run you want to inspect. Both models must contain the same completed cohort.

- **Number of connected components:** number of separate salient clusters (8-neighbor connectivity).
- **Largest component ratio:** largest cluster's cell count / all selected top-20% cells (0..1).

Use the same near-head CAMs, crop intersection and grids as E1: 7x7 primary, 14x14 sensitivity.
Select exactly ceil(0.20 * grid size) highest-valued cells, breaking ties in row-major order,
as in the existing top-20% IoU. Cutoff ties are reported because they can affect connectivity.
Only paired valid, nondegenerate maps enter model comparisons; exclusions are counted.
Fewer components and a larger ratio describe a more connected mask; they do not establish
object localization quality or causal relevance. Results remain in memory for review.


In [ ]:
import math
import numpy as np
from experiments.gradcam.experiment import RunStore
from experiments.gradcam.methods import crop_intersection, sample_grid, map_status


def _top20_components(cam):
    """Count 8-connected components of a stable exact-top-20% mask."""
    if map_status(cam) != "ok":
        raise ValueError("Connected components require a valid, nondegenerate CAM.")
    values = np.asarray(cam).ravel()
    count = math.ceil(0.2 * values.size)
    order = np.argsort(-values, kind="stable")
    mask = np.zeros(values.size, dtype=bool)
    mask[order[:count]] = True
    mask = mask.reshape(np.shape(cam))
    cutoff_tie = count < values.size and values[order[count - 1]] == values[order[count]]
    remaining = set(map(tuple, np.argwhere(mask)))
    sizes = []
    while remaining:
        pending = [remaining.pop()]
        size = 0
        while pending:
            y, x = pending.pop()
            size += 1
            neighbors = {(y + dy, x + dx) for dy in (-1, 0, 1) for dx in (-1, 0, 1)
                         if dy or dx}
            adjacent = remaining & neighbors
            remaining.difference_update(adjacent)
            pending.extend(adjacent)
        sizes.append(size)
    return len(sizes), max(sizes) / count, bool(cutoff_tie)


def _compare_components(first, second):
    """Read paired checkpoints and return small numeric arrays; never write results."""
    if (first.config["model"], second.config["model"]) != ("resnet50", "vit_b_16"):
        raise ValueError("Expected ResNet50 then ViT-B/16 checkpoints.")
    for store in (first, second):
        if store.config["experiment"] != "E1" or store.done != set(store.config["image_ids"]):
            raise ValueError("Use completed E1 checkpoints; no inference is run by this section.")
    if first.done != second.done:
        raise ValueError("The two checkpoints must contain the same image IDs.")
    for key in ("code", "shared_code", "manifest", "baseline", "dataset_config", "grids",
                "preprocessing", "dtype", "torch", "torchvision", "target"):
        if first.config[key] != second.config[key]:
            raise ValueError(f"Incompatible paired checkpoints: {key}")
    partner = {row["image_id"]: (row, maps) for row, maps in second.items()}
    values = {7: [], 14: []}
    for row, maps in first.items():
        other, other_maps = partner[row["image_id"]]
        if (row["label"], row["group"]) != (other["label"], other["group"]):
            raise ValueError("Paired labels or baseline groups differ.")
        pair = ((row, maps), (other, other_maps))
        if any("geometry" not in r or "near_head" not in m
               or r.get("map_status", {}).get("near_head") != "ok" for r, m in pair):
            continue
        region = crop_intersection(row["geometry"], other["geometry"])
        for grid in values:
            cams = [sample_grid(m["near_head"], r["geometry"], grid, region) for r, m in pair]
            if all(map_status(cam) == "ok" for cam in cams):
                values[grid].append([_top20_components(cam) for cam in cams])
    return {grid: np.asarray(rows, dtype=float).reshape(-1, 2, 3)
            for grid, rows in values.items()}


In [ ]:
# Existing run only. Change this tag to inspect another completed cohort.
COMPONENT_RUN_TAG = globals().get("RUN_TAG", "gradcam_v1_n10000")
component_root = RESULTS_DIR / "metrics/experiments/gradcam" / COMPONENT_RUN_TAG / "intermediate/E1"
component_models = ("resnet50", "vit_b_16")
if not all((component_root / model / "config.json").is_file() for model in component_models):
    print(f"No paired checkpoints found at {component_root}. Check COMPONENT_RUN_TAG; no files written.")
else:
    component_stores = [RunStore(component_root / model) for model in component_models]
    component_preview = _compare_components(*component_stores)
    print(f"Connected components | run={COMPONENT_RUN_TAG} | 8-neighbor connectivity | no files written")
    for component_grid, component_values in component_preview.items():
        component_total = len(component_stores[0].done)
        component_valid = len(component_values)
        component_k = math.ceil(0.2 * component_grid ** 2)
        print(f"\n{component_grid}x{component_grid} | top {component_k}/{component_grid ** 2} cells "
              f"| paired valid={component_valid}/{component_total}; excluded={component_total - component_valid}")
        if not component_valid:
            print("No valid pairs; metrics are undefined, not zero.")
            continue
        for component_index, component_name in enumerate(("Number of components", "Largest component ratio")):
            component_a, component_b = component_values[:, :, component_index].T
            print(f"{component_name}: ResNet mean={component_a.mean():.3f}, median={np.median(component_a):.3f}; "
                  f"ViT mean={component_b.mean():.3f}, median={np.median(component_b):.3f}")
            print(f"  Paired ViT - ResNet mean={(component_b - component_a).mean():+.3f}; "
                  f"ViT higher/equal/lower: {np.mean(component_b > component_a):.1%} / "
                  f"{np.mean(component_b == component_a):.1%} / {np.mean(component_b < component_a):.1%}")
        print(f"Cutoff ties: ResNet={component_values[:, 0, 2].mean():.1%}; "
              f"ViT={component_values[:, 1, 2].mean():.1%} (row-major tie breaking)")
    print("\nInterpretation: fewer components = fewer separated clusters; larger ratio = a larger share of selected cells in one cluster.")
    print("Compare within each grid; the 14x14 grid is a sensitivity check, not extra native ResNet resolution.")
